In [1]:
import glob
import MDAnalysis as mda

# --- File Paths ---
PDB_FILE = "start_drudes.pdb"
DCD_FILE = "FV_NVT.dcd" 

# --- Load MDAnalysis Universe ---
try:
    u = mda.Universe(PDB_FILE, DCD_FILE)
    print("✅ MDAnalysis Universe loaded successfully!")
    print(f"   - Total Atoms: {u.atoms.n_atoms}")
    print(f"   - Total Frames: {len(u.trajectory)}")
    print(f"   - Simulation Time: {len(u.trajectory) * u.trajectory.dt / 1000:.2f} ns")
except Exception as e:
    print(f"❌ File loading failed: {e}")

#### 自動讀取當前目錄下所有 .log 檔案
file_paths = glob.glob("*.log")

### 手動指定多個 .log 檔案
#file_paths = ["2V_tfsi_bmim_2cnt_CV.log", "4V_tfsi_bmim_2cnt_CV.log"]

# 儲存所有檔案的解析結果
results = {}


def _find_line(lines, keyword, start=0):
    """從 lines[start:] 找第一個包含 keyword 的行，回傳 (行號, 該行內容) 或 (None, None)"""
    for i in range(start, len(lines)):
        if keyword in lines[i]:
            return i, lines[i]
    return None, None


def _extract_value(line):
    """從 '  Kinetic:   858.289... kJ/mol' 這類行中取出數值"""
    return float(line.split()[1])


def whatever_after_normal_termination(file_path):
    with open(file_path) as f:
        lines = f.readlines()

    # 找 Initial energies
    ie_idx, _ = _find_line(lines, "Initial energies:")
    if ie_idx is None:
        print("❌ 沒有找到 Initial energies")
        return None
    # print(f"✓ 找到第一個 Initial energies 在第 {ie_idx} 行")

    # 從 Initial energies 之後找 Kinetic
    ke_idx, ke_line = _find_line(lines, "Kinetic:", start=ie_idx + 1)
    if ke_idx is None:
        print("⚠️  未找到 Kinetic energies")
        return None
    kinetic = _extract_value(ke_line)
    # print(f"✓ 找到 Kinetic energies 在第 {ke_idx} 行")
    print(f"✓ Kinetic_energies: {kinetic}")

    # 從 Initial energies 之後找 Force contributions（不必從頭找）
    fc_idx, _ = _find_line(lines, "Force contributions:", start=ie_idx + 1)
    if fc_idx is None:
        print("❌ 沒有找到 Force contributions")
        return None
    # print(f"✓ 找到第一個 Force contributions 在第 {fc_idx} 行")

    # 從 Force contributions 之後找 HarmonicBondForce 和 DrudeForce
    hb_idx, hb_line = _find_line(lines, "HarmonicBondForce:", start=fc_idx + 1)
    df_idx, df_line = _find_line(lines, "DrudeForce:", start=fc_idx + 1)

    if hb_idx is None:
        print("⚠️  未找到 HarmonicBondForce")
        return None
    harmonic = _extract_value(hb_line)
    # print(f"✓ 找到 HarmonicBondForce 在第 {hb_idx} 行")
    print(f"✓ HarmonicBondForce: {harmonic}")

    if df_idx is None:
        print("⚠️  未找到 DrudeForce")
        return None
    drude = _extract_value(df_line)
    # print(f"✓ 找到 DrudeForce 在第 {df_idx} 行")
    print(f"✓ DrudeForce: {drude}")

    return {
        'Kinetic_energies': kinetic,
        'HarmonicBondForce': harmonic,
        'DrudeForce': drude,
    }


# 處理所有找到的 .log 檔案
if file_paths:
    print(f"\n找到 {len(file_paths)} 個 .log 檔案")
    print("=" * 80)

    failed_files = []

    for file_path in file_paths:
        print(f"\n📄 處理檔案: {file_path}")
        print("=" * 80)
        result = whatever_after_normal_termination(file_path)
        if result is None:
            failed_files.append(file_path)
        else:
            results[file_path] = result
        print("\n" + "=" * 80)

    # 總結報告
    print("\n" + "=" * 80)
    print("📊 處理結果總結")
    print("=" * 80)
    print(f"✓ 成功: {len(results)} 個檔案")
    print(f"✗ 失敗: {len(failed_files)} 個檔案")

    if failed_files:
        print("\n失敗的檔案列表:")
        for f in failed_files:
            print(f"  - {f}")

else:
    print("未找到任何 .log 檔案")

/home/atuo/miniforge3/envs/cuda/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


❌ File loading failed: [Errno 2] No such file or directory: 'start_drudes.pdb'
未找到任何 .log 檔案


In [2]:
import glob
import re
import MDAnalysis as mda

# --- 1. MDAnalysis 部分 (保持原樣，但加上檢查檔案是否存在) ---
# 建議把這部分封裝，不要散落在主程式
def check_trajectory(pdb, dcd):
    try:
        u = mda.Universe(pdb, dcd)
        print(f"✅ Trajectory Loaded: {u.atoms.n_atoms} atoms, "
              f"{len(u.trajectory)} frames, "
              f"{len(u.trajectory) * u.trajectory.dt / 1000:.2f} ns")
        return True
    except Exception as e:
        print(f"⚠️ Trajectory Warning: {e} (Skipping MDA check)")
        return False

# --- 2. 解析 Log 的核心邏輯 (大幅瘦身) ---
def parse_log_file(file_path):
    """
    只需遍歷一次檔案，即可抓取所有需要的數值。
    """
    # 定義你想抓的關鍵字 (Log中的文字 : 存起來的Key)
    target_map = {
        "Kinetic:": "Kinetic_energies",
        "HarmonicBondForce:": "HarmonicBondForce",
        "DrudeForce:": "DrudeForce"
    }
    
    data = {}
    found_start = False # 狀態鎖：是否已經讀到 Initial energies

    try:
        with open(file_path, 'r') as f:
            for line in f:
                # 1. 檢查是否進入目標區塊
                if "Initial energies:" in line:
                    found_start = True
                    continue # 跳過這一行，繼續看下一行

                # 2. 如果還沒進入區塊，就跳過
                if not found_start:
                    continue

                # 3. 掃描這一行是否包含我們要的關鍵字
                for log_key, out_key in target_map.items():
                    if log_key in line:
                        # 使用 Regex 抓取數值 (支援整數、浮點數、科學記號)
                        # 邏輯：抓取冒號後面的第一個數字
                        match = re.search(r"[-+]?\d*\.\d+|[-+]?\d+", line)
                        if match:
                            data[out_key] = float(match.group())
                
                # 優化：如果找齊了所有數據，就可以提早結束讀取 (Optional)
                if len(data) == len(target_map):
                    break
        
        # 檢查是否所有數據都找到了
        if len(data) != len(target_map):
            missing = set(target_map.values()) - set(data.keys())
            print(f"⚠️ {file_path}: 缺少數據 {missing}")
            return None # 視為失敗

        return data

    except Exception as e:
        print(f"❌ 讀取錯誤 {file_path}: {e}")
        return None

# --- 3. 主執行區塊 ---
if __name__ == "__main__":
    # 檢查 Trajectory (Optional)
    check_trajectory("start_drudes.pdb", "FV_NVT.dcd")

    # 取得檔案列表
    file_paths = glob.glob("*.log")
    
    if not file_paths:
        print("❌ 未找到任何 .log 檔案")
    else:
        results = {}
        print(f"\n🔍 開始分析 {len(file_paths)} 個檔案...\n")
        
        for fp in file_paths:
            res = parse_log_file(fp)
            if res:
                results[fp] = res
                # 簡潔的輸出
                print(f"📄 {fp:<30} | Kinetic: {res['Kinetic_energies']:.2f} | "
                      f"Harmonic: {res['HarmonicBondForce']:.2f} | Drude: {res['DrudeForce']:.2f}")
        
        print("\n" + "="*60)
        print(f"📊 總結: 成功 {len(results)} / 失敗 {len(file_paths) - len(results)}")

⚠️ Trajectory Warning: [Errno 2] No such file or directory: 'start_drudes.pdb' (Skipping MDA check)
❌ 未找到任何 .log 檔案
